# 04 · Character n-gram model on the raw URL

**Idea:** instead of deciding on features ourselves, we let the model see every short chunk of characters in the URL.

A **character n-gram** is a run of *n* consecutive characters. The URL `paypa1.top` contains the 3-grams `pay`, `ayp`, `ypa`, `pa1`, `a1.`, `1.t`, `.to`, `top`. Each URL becomes a long, mostly-zero vector holding a score for every n-gram that appears in it:
- **TF-IDF** weights n-grams that are common in this URL but rare overall.
- **Logistic Regression** learns one weight per n-gram, pushing towards phishing or towards legitimate.

This is the classical forerunner of the character-level CNN in Milestone 4: it catches typosquats (`lcioud`, `ncsbarnk`), cheap TLDs (`.top`) and odd character mixes without hand-written rules.

Published URL-only references (Frontiers in Computer Science, 2026): Logistic Regression 99.62%, CharCNN 99.81%.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier

from phishguard.data import load_clean, TARGET
from phishguard.split import get_splits
from phishguard.features import extract_features
from phishguard.modeling import param_grid, tune, fit_and_test
from phishguard.evaluate import best_threshold, evaluate, save_result

df = load_clean()
splits = get_splits(df)
train_idx, val_idx, test_idx = splits
y = df[TARGET]
urls = df["URL"]

## 1. Tune on validation

Two settings matter:
- `max_n`: the longest n-gram used (from 1 up to `max_n`). Longer n-grams capture whole words like `login`, but there are many more of them.
- `C`: regularisation strength (smaller means simpler).

In [2]:
def make_ngram(max_n, C):
    return make_pipeline(
        TfidfVectorizer(analyzer="char", ngram_range=(1, max_n), min_df=2,
                        sublinear_tf=True, dtype=np.float32),
        LogisticRegression(C=C, solver="liblinear", max_iter=3000),
    )

table, best = tune(make_ngram, param_grid(max_n=[3, 4, 5], C=[3.0, 10.0]), urls, y, splits)
table

,max_n,C,val_accuracy,val_roc_auc
0,4,10.0,0.997723,0.999336
1,3,10.0,0.997723,0.999301
2,5,10.0,0.997723,0.999178
3,3,3.0,0.997609,0.999282
4,4,3.0,0.997523,0.999278
5,5,3.0,0.997495,0.999101


In [3]:
ngram, t_ngram, m = fit_and_test(make_ngram(**best), urls, y, splits)
save_result("Char n-gram LogReg (raw URL)", "raw URL", t_ngram, m)
vectorizer, logreg = ngram.named_steps.values()
print(f"{len(vectorizer.vocabulary_):,} distinct n-grams")
pd.Series(m).round(5)

308,150 distinct n-grams


accuracy         0.99695
precision        0.99879
recall           0.99403
f1               0.99640
roc_auc          0.99874
tn           20205.00000
fp              18.00000
fn              89.00000
tp           14818.00000
dtype: float64

## 2. What did it learn?

Each n-gram has one weight. Large positive weights push towards phishing, large negative weights towards legitimate.

In [4]:
weights = pd.Series(logreg.coef_[0], index=vectorizer.get_feature_names_out())
pd.DataFrame({
    "→ phishing": weights.nlargest(20).index,
    "weight": weights.nlargest(20).round(1).values,
    "→ legitimate": weights.nsmallest(20).index,
    "weight ": weights.nsmallest(20).round(1).values,
})

,→ phishing,weight,→ legitimate,weight
0,p://,25.8,ww,-24.5
1,p:,25.8,w,-22.5
2,p:/,25.8,www.,-16.1
3,tp:,25.8,ww.,-15.6
4,tp:/,25.8,/www,-14.9
5,ttp:,25.8,www,-14.8
6,m/,11.8,//ww,-14.7
7,com/,11.7,/ww,-14.6
8,om/,11.7,w.,-14.3
9,.top,8.5,://w,-13.2


**Look at the top of both lists.** The strongest phishing signal is `p://`, which is `http://` *without* the `s`. The strongest legitimate signals are `www` and `s://`. The model has learned mostly *how the URL starts*, not what the domain says. Every legitimate URL in PhiUSIIL starts with `https://www.`, so this works on our test set. A real phishing site can use `https://www.` too, though. This is the main question for Milestone 3. Further down the lists you can see genuine signals such as `.top`.

## 3. Combining with the URL-feature model

The two models make different mistakes: XGBoost sees counts and structure, while the n-gram model sees the actual characters. Averaging their probabilities is the simplest combination. It's a first preview of the hybrid model in Milestone 5.

The XGBoost settings are the best ones from notebook 03.

In [5]:
features = extract_features(urls)
xgb, _, _ = fit_and_test(
    XGBClassifier(max_depth=10, n_estimators=800, learning_rate=0.1, n_jobs=-1, random_state=42),
    features, y, splits)

def averaged(idx):
    return (ngram.predict_proba(urls.loc[idx])[:, 1] + xgb.predict_proba(features.loc[idx])[:, 1]) / 2

t_avg = best_threshold(y.loc[val_idx], averaged(val_idx))
m = evaluate(y.loc[test_idx], averaged(test_idx), t_avg)
save_result("Average: n-gram + XGBoost (URL only)", "raw URL + URL features", t_avg, m)
pd.Series(m).round(5)

accuracy         0.99707
precision        0.99926
recall           0.99383
f1               0.99654
roc_auc          0.99859
tn           20212.00000
fp              11.00000
fn              92.00000
tp           14815.00000
dtype: float64

## Result

The URL-only models land at about 99.7–99.8%, in line with the best published URL-only results on PhiUSIIL. What's left is mostly phishing sites that look like a plain `https://www.<domain>` homepage, where the domain name is the only clue (see notebook 03).

The comparison table across all models is in notebook 05.